# Phase 2: Data Cleaning & Standardization


In [1]:
import pandas as pd

final_df = pd.read_csv(
    "../data/processed/parsed_messages.csv",
    parse_dates=["date"]
)

final_df.shape

(4596, 18)

In [ ]:
# Remove Invalid Parsed Records:

final_df = final_df[final_df["parse_error"] == False].reset_index(drop=True)

final_df.shape


(4548, 18)

In [ ]:
# Missing Value Analysis:

final_df.isna().mean().sort_values(ascending=False)

professor_id             1.000000
term                     0.060026
comment_text             0.060026
rating_6                 0.059807
grading_status_raw       0.059807
rating_3                 0.059807
rating_5                 0.059807
rating_4                 0.059807
rating_2                 0.059807
rating_1                 0.059807
attendance_status_raw    0.059807
department               0.000220
date                     0.000000
id                       0.000000
course_name              0.000000
professor_name_raw       0.000000
date_unixtime            0.000000
parse_error              0.000000
dtype: float64

In [ ]:
# Drop Rows with No Ratings:

rating_cols = [f"rating_{i}" for i in range(1, 7)]

final_df = final_df.dropna(subset=rating_cols, how="all")

final_df.shape

(4276, 18)

## Text Standardization

To improve consistency and prevent duplicate representations, textual fields were standardized.  
This includes normalization of Persian characters and removal of minor formatting variations in professor names.


In [ ]:
# Standardize Professor Names:

final_df["professor_name"] = (
    final_df["professor_name_raw"]
    .str.strip()
    .str.replace("ي", "ی")
    .str.replace("ك", "ک")
)

In [ ]:
# Generate Professor ID:

final_df["professor_id"] = (
    final_df["professor_name"]
    .astype("category")
    .cat.codes
)

In [7]:
for col in rating_cols:
    final_df[col] = pd.to_numeric(final_df[col], errors="coerce")

In [8]:
for col in rating_cols:
    final_df.loc[
        (final_df[col] < 0) | (final_df[col] > 10), col
    ] = None


In [ ]:
# Create Overall Rating:

final_df["rating_mean"] = final_df[rating_cols].mean(axis=1)

In [ ]:
# Faculty / Department Extraction:

def extract_department(text):
    if text is None:
        return None
    if "ریاضی" in text:
        return "Mathematics"
    if "فیزیک" in text:
        return "Physics"
    if "کامپیوتر" in text:
        return "Computer Engineering"
    return None

final_df["department_std"] = final_df["course_name"].apply(extract_department)


Course name standardization and advanced textual labeling were intentionally limited to avoid heuristic-driven errors, as no authoritative reference dictionary was provided. Only minimal and conservative labeling was applied where the signal was clear.

In [11]:
final_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 4276 entries, 0 to 4547
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype         
---  ------                 --------------  -----         
 0   id                     4276 non-null   int64         
 1   date                   4276 non-null   datetime64[ns]
 2   date_unixtime          4276 non-null   int64         
 3   professor_id           4276 non-null   int16         
 4   professor_name_raw     4276 non-null   object        
 5   department             4275 non-null   object        
 6   course_name            4276 non-null   object        
 7   rating_1               4276 non-null   float64       
 8   rating_2               4276 non-null   float64       
 9   rating_3               4276 non-null   float64       
 10  rating_4               4276 non-null   float64       
 11  rating_5               4276 non-null   float64       
 12  rating_6               4276 non-null   float64       
 13  grading_

## Standardization of Grading and Attendance Status

Grading and attendance information is provided in free-text form and exhibits high variability.  
To ensure consistency while avoiding heuristic bias, these fields were conservatively standardized into a small set of categorical labels.  
Ambiguous cases were intentionally labeled as `unknown`.

In [16]:
def standardize_grading(text):
    if text is None:
        return "unknown"

    text = text.replace("ي", "ی").replace("ك", "ک")

    if any(k in text for k in ["منصفانه"]):
        return "fair"
    if any(k in text for k in ["نمره خوبی نمیشه ازشون گرفت"]):
        return "strict"
    if any(k in text for k in ["دست باز و با ارفاق"]):
        return "lenient"

    return "unknown"


final_df["grading_status_std"] = (
    final_df["grading_status_raw"]
    .apply(standardize_grading)
)


In [18]:
def standardize_attendance(text):
    if text is None:
        return "unknown"

    text = text.replace("ي", "ی").replace("ك", "ک")

    if any(k in text for k in ["حضور مهم است", "تاثیر مستقیم دارد"]):
        return "mandatory"
    if "حضور مهم نیست" in text and "اما تاثیر مثبت دارد" in text:
        return "positive_effect"
    if "حضور و غیاب نمی کند" in text:
        return "optional"

    return "unknown"


final_df["attendance_status_std"] = (
    final_df["attendance_status_raw"]
    .apply(standardize_attendance)
)


In [20]:
final_df[[
    "grading_status_raw",
    "grading_status_std",
    "attendance_status_raw",
    "attendance_status_std"
]].sample(10, random_state=42)

,grading_status_raw,grading_status_std,attendance_status_raw,attendance_status_std
563,منصفانه و هرچی خودت بگیری,fair,حضور و غیاب نمی کند,optional
416,دست باز و با ارفاق,lenient,حضور و غیاب نمی کند,optional
2814,منصفانه و هرچی خودت بگیری,fair,حضور مهم است و تاثیر مستقیم دارد,mandatory
1344,نمره خوبی نمیشه ازشون گرفت,strict,حضور و غیاب نمی کند,optional
623,دست باز و با ارفاق,lenient,حضور مهم نیست اما تاثیر مثبت دارد,positive_effect
2352,دست باز و با ارفاق,lenient,حضور مهم است و تاثیر مستقیم دارد,mandatory
4432,منصفانه و هرچی خودت بگیری,fair,حضور مهم است و تاثیر مستقیم دارد,mandatory
3788,منصفانه و هرچی خودت بگیری,fair,حضور مهم است و تاثیر مستقیم دارد,mandatory
4412,نمره خوبی نمیشه ازشون گرفت,strict,حضور مهم است و تاثیر مستقیم دارد,mandatory
2562,منصفانه و هرچی خودت بگیری,fair,حضور و غیاب نمی کند,optional


## Phase 2 Summary

In this phase, the parsed dataset was transformed into a clean and analysis-ready format.  
Invalid and non-informative records were removed, textual and numeric fields were standardized, and essential analytical features were created.

The resulting dataset is consistent, reliable, and suitable for exploratory analysis and modeling.
